# Classifying Heart Disease from BRFSS Data

This notebook reproduces all experiments, cross-validation comparisons, and ablation studies presented in [`project_1_report.pdf`](project_1_report.pdf). All preprocessing and validation routines are imported from `preprocessing.py` and `validation.py` (the same modules used by `run.py`).

## 1. Introduction & Exploratory Data Analysis

The 2015 BRFSS training dataset contains **328,135 respondents** and **321 features**, with **8.83%** belonging to the positive class (`_MICHD = 1`, history of myocardial infarction or coronary heart disease).
- **Metric choice:** Because of class imbalance, predicting every respondent as negative achieves **91.17% accuracy** while detecting zero cases ($F_1 = 0$). We therefore optimize $F_1 = \frac{2PR}{P+R}$ and also track **AUC**, which measures patient ranking independently of the decision threshold.
- **Missingness & skip patterns:** Every respondent has at least 25% missing values (median 43%), so missingness must be handled column-wise. Most missing values arise from questionnaire skip patterns (e.g., `HAREHAB1`, cardiac rehabilitation, is only asked to respondents who reported a heart attack and has a 100% positive rate, making it a target-leakage column).
- **Non-monotone & coded responses:** Special numeric codes represent `"don't know"`/`"refused"` (`7, 9, 77, 99, ...`) or `"none"` (`88, 555, 888`). Furthermore, some categorical variables carry non-monotone risk (e.g., in `_SMOKER3`, former smokers have a higher positive rate than current smokers) while the observed positive rate increases with age.

In [1]:
import time
import numpy as np

import preprocessing as pp
from helpers import load_csv_data
from implementations import (
    least_squares,
    logistic_regression,
    mean_squared_error_gd,
    mean_squared_error_sgd,
    reg_logistic_regression,
    ridge_regression,
)
from validation import (
    SEED,
    accuracy,
    auc_score,
    corrected_se,
    cross_validate,
    f1_score,
    print_cv,
    stratified_kfold_indices,
)

DATA_PATH = "dataset/"
K = 5


def sigmoid(t):
    """Logistic function."""
    return 1 / (1 + np.exp(-t))


x_train, x_test, y_train, train_ids, test_ids = load_csv_data(DATA_PATH)
col_names = pp.read_column_names(DATA_PATH + "x_train.csv")
col_idx = {name: j for j, name in enumerate(col_names)}
y01 = ((y_train + 1) / 2).astype(int)

# Dataset dimensions & trivial baselines
p = np.mean(y01 == 1)
print(f"x_train: {x_train.shape} | x_test: {x_test.shape} | positive prevalence p = {p:.4f}")
print(f"Always negative -> accuracy: {1 - p:.4f} | F1: 0.0000")
print(f"Always positive -> accuracy: {p:.4f} | F1: {2 * p / (1 + p):.4f} | AUC: 0.5000")

# Row-wise missingness & HAREHAB1 target leakage check
nan_rows = np.isnan(x_train).mean(axis=1)
print(f"NaN per row     -> min: {nan_rows.min():.3f} | median: {np.median(nan_rows):.3f} | max: {nan_rows.max():.3f}")
answered = ~np.isnan(x_train[:, col_idx["HAREHAB1"]])
print(f"HAREHAB1 answered: {answered.sum()} | positive rate among answered: {y01[answered].mean():.3f}")

# Positive rate by age group (_AGEG5YR) and smoking status (_SMOKER3)
for name, max_valid in [("_AGEG5YR", 13), ("_SMOKER3", 4)]:
    col = x_train[:, col_idx[name]]
    parts = [f"{v:g}: {y01[col == v].mean():.3f}" for v in np.unique(col[col <= max_valid])]
    print(f"{name:<8} -> " + " | ".join(parts))

x_train: (328135, 321) | x_test: (109379, 321) | positive prevalence p = 0.0883
Always negative -> accuracy: 0.9117 | F1: 0.0000
Always positive -> accuracy: 0.0883 | F1: 0.1623 | AUC: 0.5000
NaN per row     -> min: 0.249 | median: 0.433 | max: 0.679
HAREHAB1 answered: 773 | positive rate among answered: 1.000
_AGEG5YR -> 1: 0.005 | 2: 0.009 | 3: 0.011 | 4: 0.015 | 5: 0.023 | 6: 0.037 | 7: 0.056 | 8: 0.072 | 9: 0.100 | 10: 0.123 | 11: 0.161 | 12: 0.185 | 13: 0.222
_SMOKER3 -> 1: 0.104 | 2: 0.098 | 3: 0.135 | 4: 0.062


## 2. Methods — Preprocessing & Validation Protocol

### 2.1 Column Exclusion & Recoding Special Codes
Following the 2015 BRFSS codebook, `preprocessing.py` applies five steps:
1. **Excluding columns (103 of 321):** 31 administrative variables, 2 date columns, the state code (`_STATE`), 1 leakage column (`HAREHAB1`), and 68 derived or substantially overlapping variables excluded to simplify the model, leaving **218 columns**.
2. **Recoding special codes:** `"don't know"` and `"refused"` (`7, 9, 77, 99, 777, 999`) are mapped to `NaN`, and `"none"`/`"never"` (`88, 555, 888`) are mapped to `0`, with codebook-verified exceptions (drink counts `1`–`76` are valid, while `77` means 'don't know'; `88` is a valid diabetes diagnosis age in `DIABAGE2`).

In [2]:
for name, group in [("administrative", pp.ADMIN), ("leakage", pp.LEAKAGE),
                    ("redundant", pp.REDUNDANT), ("dates", pp.DATES), ("not used", pp.NOT_USED)]:
    print(f"{name:<15} {len(group):>3}")

kept_names, recode = pp.recoding_rules(x_train, col_names)
xr_train = pp.apply_recoding(x_train, col_names, kept_names, recode)
xr_test = pp.apply_recoding(x_test, col_names, kept_names, recode)
kept = np.array([j for j, name in enumerate(col_names) if name not in pp.EXCLUDED])
before = x_train[:, kept]

print(f"\nExcluded: {len(pp.EXCLUDED)} | kept: {len(kept_names)}")
print(f"Values mapped to NaN: {np.sum(~np.isnan(before) & np.isnan(xr_train))}")
print(f"Values mapped to 0:   {np.sum((before != 0) & (xr_train == 0))}")

administrative   31
leakage           1
redundant        68
dates             2
not used          1

Excluded: 103 | kept: 218
Values mapped to NaN: 873198
Values mapped to 0:   739001


### 2.2 Stratified 5-Fold Cross-Validation & Threshold Selection
We evaluate the main model configurations using **stratified 5-fold cross-validation** (`SEED = 42`). Inside each fold, we select the $F_1$-maximising decision threshold on an inner fifth of the training split, then refit on the full training split and evaluate on the held-out fold.

To compare two configurations on the same folds, we compute their paired per-fold differences $d_i$ and correct the standard error for the 75% overlap between training sets (Nadeau & Bengio):
$$\widehat{\mathrm{SE}} = \sqrt{\left(\frac{1}{k} + \frac{n_{\text{test}}}{n_{\text{train}}}\right) s_d^2}, \qquad \frac{n_{\text{test}}}{n_{\text{train}}} = \frac{1}{k - 1}.$$
**Selection rule:** A modification is adopted only if it improves $F_1$ by at least $2\,\widehat{\mathrm{SE}}$ and $0.002$, without lowering AUC by more than $2\,\widehat{\mathrm{SE}}$.

Below we set up the folds and evaluate the **age + sex logistic regression baseline**.

In [3]:
folds = stratified_kfold_indices(y01, K, SEED)


def logreg_age_sex(x_tr, y_tr, x_va):
    """Logistic regression on standardized age and sex."""
    cols = [col_idx["_AGE80"], col_idx["SEX"]]
    mean, std = x_tr[:, cols].mean(axis=0), x_tr[:, cols].std(axis=0)
    tx_tr = np.c_[np.ones(len(x_tr)), (x_tr[:, cols] - mean) / std]
    tx_va = np.c_[np.ones(len(x_va)), (x_va[:, cols] - mean) / std]
    w, _ = logistic_regression(y_tr, tx_tr, np.zeros(tx_tr.shape[1]), max_iters=500, gamma=0.5)
    return sigmoid(tx_va @ w)


f1s, accs, aucs, ths = cross_validate(x_train, y01, folds, logreg_age_sex)
print_cv("LogReg (age, sex)", f1s, accs, aucs, ths)

LogReg (age, sex)      F1: 0.2935 ± 0.0040 | acc: 0.7887 ± 0.0078 | AUC: 0.7529 ± 0.0033 | threshold: 0.148 ± 0.004


### 2.3 Handling Missing Values (Selecting $\tau$) & One-Hot Encoding
3. **Handling missing values:** We retain columns with at most a fraction $\tau$ of missing values, impute the training median, and add one binary indicator per distinct missingness pattern. We select $\tau$ over the grid $\{0.1, 0.3, 0.5, 0.8, 0.9\}$ by cross-validation.
4. **Encoding and scaling:** For **23 nominal or non-monotone ordinal variables** (`pp.NOMINAL`), we one-hot encode them using $k - 1$ dummies to avoid collinearity with the bias. All features are standardized using the training fold mean and standard deviation.
5. **Optimization:** The final unregularized logistic model starts from $w = 0$ and uses step size $\gamma = 1/L$, where $L = \lambda_{\max}(X^\top X / n)/4$ is the Lipschitz constant of the logistic loss gradient, for **1000 iterations**.

In [4]:
def make_logreg(tau, nominal=frozenset(), continuous=frozenset(), max_iters=1000):
    """Preprocessing + logistic regression with step size 1/L."""
    def train_and_predict(x_tr, y_tr, x_va):
        prep = pp.fit_preprocessing(x_tr, tau, nominal, continuous)
        tx_tr, tx_va = pp.transform(x_tr, prep), pp.transform(x_va, prep)
        L = np.linalg.eigvalsh(tx_tr.T @ tx_tr / len(tx_tr)).max() / 4
        w, _ = logistic_regression(y_tr, tx_tr, np.zeros(tx_tr.shape[1]), max_iters, 1 / L)
        return sigmoid(tx_va @ w)
    return train_and_predict


results = {}
for tau in [0.1, 0.3, 0.5, 0.8, 0.9]:
    results[f"tau={tau}"] = cross_validate(xr_train, y01, folds, make_logreg(tau))
    print_cv(f"tau={tau}", *results[f"tau={tau}"])

print()
best = results["tau=0.9"]
for tau in [0.3, 0.5, 0.8]:
    d_f1 = best[0] - results[f"tau={tau}"][0]
    d_auc = best[2] - results[f"tau={tau}"][2]
    se_f1, se_auc = corrected_se(d_f1, K), corrected_se(d_auc, K)
    print(f"0.9 vs {tau} -> dF1: {d_f1.mean():+.4f} (SE {se_f1:.4f}, {d_f1.mean() / se_f1:.1f} SE) | "
          f"dAUC: {d_auc.mean():+.4f} (SE {se_auc:.4f}, {d_auc.mean() / se_auc:.1f} SE)")

tau=0.1                F1: 0.4133 ± 0.0027 | acc: 0.8628 ± 0.0054 | AUC: 0.8517 ± 0.0004 | threshold: 0.185 ± 0.008
tau=0.3                F1: 0.4215 ± 0.0037 | acc: 0.8633 ± 0.0067 | AUC: 0.8578 ± 0.0010 | threshold: 0.185 ± 0.011
tau=0.5                F1: 0.4214 ± 0.0027 | acc: 0.8696 ± 0.0073 | AUC: 0.8578 ± 0.0009 | threshold: 0.197 ± 0.013
tau=0.8                F1: 0.4250 ± 0.0038 | acc: 0.8714 ± 0.0016 | AUC: 0.8586 ± 0.0008 | threshold: 0.199 ± 0.004
tau=0.9                F1: 0.4258 ± 0.0034 | acc: 0.8748 ± 0.0018 | AUC: 0.8589 ± 0.0010 | threshold: 0.206 ± 0.006

0.9 vs 0.3 -> dF1: +0.0042 (SE 0.0023, 1.8 SE) | dAUC: +0.0011 (SE 0.0003, 4.0 SE)
0.9 vs 0.5 -> dF1: +0.0044 (SE 0.0026, 1.7 SE) | dAUC: +0.0011 (SE 0.0002, 4.4 SE)
0.9 vs 0.8 -> dF1: +0.0008 (SE 0.0013, 0.6 SE) | dAUC: +0.0003 (SE 0.0002, 1.6 SE)


In [5]:
nominal_idx = pp.column_indices(kept_names, pp.NOMINAL)

# One-hot encoding of the 23 nominal / non-monotone variables at tau = 0.8
results["one-hot"] = cross_validate(xr_train, y01, folds, make_logreg(pp.TAU, nominal_idx))
print_cv("one-hot", *results["one-hot"])

ref = results[f"tau={pp.TAU}"]
d_f1 = results["one-hot"][0] - ref[0]
d_auc = results["one-hot"][2] - ref[2]
print(f"one-hot vs numeric -> dF1: {d_f1.mean():+.4f} (SE {corrected_se(d_f1, K):.4f}) | "
      f"dAUC: {d_auc.mean():+.4f} (SE {corrected_se(d_auc, K):.4f})")

# Final design matrix breakdown at tau = 0.8
prep = pp.fit_preprocessing(xr_train, pp.TAU, nominal_idx)
n_num = len(prep["num_cols"])
n_dum = sum(len(c) for c in prep["categories"])
n_ind = len(prep["ind_cols"])
print(f"\nSurvey columns kept: {n_num + len(prep['cat_cols'])} -> {n_num} numeric + {n_dum} dummies + "
      f"{n_ind} missingness indicators + 1 bias = {pp.transform(xr_train[:5], prep).shape[1]} features")

# Convergence check on Fold 0 (500 vs 1000 vs 2000 iterations)
va, tr = folds[0], np.concatenate(folds[1:])
y_tr, y_va = y01[tr], y01[va]
prep0 = pp.fit_preprocessing(xr_train[tr], pp.TAU, nominal_idx)
tx_tr, tx_va = pp.transform(xr_train[tr], prep0), pp.transform(xr_train[va], prep0)
gamma = 4 / np.linalg.eigvalsh(tx_tr.T @ tx_tr / len(tx_tr)).max()

w, done = np.zeros(tx_tr.shape[1]), 0
for total in [500, 1000, 2000]:
    w, loss = logistic_regression(y_tr, tx_tr, w, total - done, gamma)
    done = total
    grad = tx_tr.T @ (sigmoid(tx_tr @ w) - y_tr) / len(y_tr)
    print(f"iters {total:>5} | train loss {loss:.5f} | grad norm {np.linalg.norm(grad):.2e} | "
          f"val AUC {auc_score(y_va, sigmoid(tx_va @ w)):.4f}")

one-hot                F1: 0.4275 ± 0.0031 | acc: 0.8701 ± 0.0030 | AUC: 0.8600 ± 0.0008 | threshold: 0.196 ± 0.008
one-hot vs numeric -> dF1: +0.0025 (SE 0.0007) | dAUC: +0.0013 (SE 0.0002)

Survey columns kept: 106 -> 83 numeric + 95 dummies + 98 missingness indicators + 1 bias = 277 features
iters   500 | train loss 0.22214 | grad norm 5.69e-03 | val AUC 0.8604
iters  1000 | train loss 0.22109 | grad norm 2.16e-03 | val AUC 0.8612
iters  2000 | train loss 0.22081 | grad norm 7.51e-04 | val AUC 0.8615


- **Choice of $\tau = 0.8$:** $\tau = 0.9$ achieves the highest score, but its gain over $\tau = 0.8$ is only $0.6\,\widehat{\mathrm{SE}}$ on $F_1$ ($+0.0008$), so by our selection rule we retain the simpler $\tau = 0.8$ model.
- **One-hot encoding:** Improves $F_1$ by $+0.0025$ ($3.6\,\widehat{\mathrm{SE}}$) and AUC by $+0.0013$ ($6.5\,\widehat{\mathrm{SE}}$), turning **106 survey questions** into **277 features** (83 numeric, 95 dummies, 98 missingness indicators, and the bias).
- **Iterations:** Doubling the iterations from 1000 to 2000 changes validation AUC by only $3 \cdot 10^{-4}$.

## 3. Results — Comparison of Regression & Classification Methods (Table I)

Least squares, ridge, logistic and regularized logistic regression use the same five-fold protocol. GD and SGD are checked against the least-squares solution on fold 0, using 3,000 and 3,000,000 updates respectively.

In [6]:
def make_linear(fit):
    """Final preprocessing + linear model fitted by fit(y, tx) -> (w, loss)."""
    def train_and_predict(x_tr, y_tr, x_va):
        prep = pp.fit_preprocessing(x_tr, pp.TAU, nominal_idx)
        w, _ = fit(y_tr, pp.transform(x_tr, prep))
        return pp.transform(x_va, prep) @ w
    return train_and_predict


results5 = {"logistic": results["one-hot"]}
results5["least squares"] = cross_validate(xr_train, y01, folds, make_linear(least_squares))
print_cv("least squares", *results5["least squares"])

ref = results5["logistic"]
d_f1 = results5["least squares"][0] - ref[0]
d_auc = results5["least squares"][2] - ref[2]
print(f"least squares vs logistic -> dF1: {d_f1.mean():+.4f} (SE {corrected_se(d_f1, K):.4f}) | "
      f"dAUC: {d_auc.mean():+.4f} (SE {corrected_se(d_auc, K):.4f})\n")

# Ridge regression grid
lambdas = np.logspace(-6, 1, 8)
ref_ls = results5["least squares"]
for lam in lambdas:
    name = f"ridge {lam:.0e}"
    results5[name] = cross_validate(xr_train, y01, folds, make_linear(lambda y, tx: ridge_regression(y, tx, lam)))
    print_cv(name, *results5[name])

least squares          F1: 0.4140 ± 0.0020 | acc: 0.8571 ± 0.0060 | AUC: 0.8518 ± 0.0005 | threshold: 0.204 ± 0.007
least squares vs logistic -> dF1: -0.0135 (SE 0.0020) | dAUC: -0.0082 (SE 0.0002)

ridge 1e-06            F1: 0.4140 ± 0.0020 | acc: 0.8571 ± 0.0060 | AUC: 0.8518 ± 0.0005 | threshold: 0.204 ± 0.007
ridge 1e-05            F1: 0.4141 ± 0.0020 | acc: 0.8571 ± 0.0059 | AUC: 0.8518 ± 0.0005 | threshold: 0.204 ± 0.007
ridge 1e-04            F1: 0.4141 ± 0.0022 | acc: 0.8571 ± 0.0060 | AUC: 0.8518 ± 0.0005 | threshold: 0.204 ± 0.007
ridge 1e-03            F1: 0.4147 ± 0.0026 | acc: 0.8614 ± 0.0051 | AUC: 0.8518 ± 0.0005 | threshold: 0.209 ± 0.006
ridge 1e-02            F1: 0.4145 ± 0.0025 | acc: 0.8624 ± 0.0050 | AUC: 0.8521 ± 0.0004 | threshold: 0.208 ± 0.007
ridge 1e-01            F1: 0.4142 ± 0.0032 | acc: 0.8638 ± 0.0067 | AUC: 0.8529 ± 0.0003 | threshold: 0.193 ± 0.007
ridge 1e+00            F1: 0.4019 ± 0.0034 | acc: 0.8593 ± 0.0045 | AUC: 0.8478 ± 0.0005 | threshold: 0.1

In [7]:
def make_reg_logreg(lam, max_iters=1000):
    """Final preprocessing + L2-regularized logistic regression with step size 1/L."""
    def train_and_predict(x_tr, y_tr, x_va):
        prep = pp.fit_preprocessing(x_tr, pp.TAU, nominal_idx)
        tx_tr, tx_va = pp.transform(x_tr, prep), pp.transform(x_va, prep)
        L = np.linalg.eigvalsh(tx_tr.T @ tx_tr / len(tx_tr)).max() / 4 + 2 * lam
        w, _ = reg_logistic_regression(y_tr, tx_tr, lam, np.zeros(tx_tr.shape[1]), max_iters, 1 / L)
        return sigmoid(tx_va @ w)
    return train_and_predict


ref = results5["logistic"]
for lam in np.logspace(-6, -2, 5):
    name = f"reg logistic {lam:.0e}"
    results5[name] = cross_validate(xr_train, y01, folds, make_reg_logreg(lam))
    print_cv(name, *results5[name])

# Verify that MSE GD and SGD converge to the closed-form least squares solution (Fold 0)
print()
w_ls, loss_ls = least_squares(y_tr, tx_tr)
print(f"{'least squares':<14} | loss {loss_ls:.6f} | val AUC {auc_score(y_va, tx_va @ w_ls):.4f}")

gamma_gd = 1 / np.linalg.eigvalsh(tx_tr.T @ tx_tr / len(tx_tr)).max()
w_gd, loss_gd = mean_squared_error_gd(y_tr, tx_tr, np.zeros(tx_tr.shape[1]), 3000, gamma_gd)
print(f"{'GD 3000':<14} | loss {loss_gd:.6f} | ||w - w_ls|| {np.linalg.norm(w_gd - w_ls):.2e} | "
      f"val AUC {auc_score(y_va, tx_va @ w_gd):.4f}")

np.random.seed(SEED)
gamma_sgd = 1 / np.sum(tx_tr ** 2, axis=1).max()
w_sgd, loss_sgd = mean_squared_error_sgd(y_tr, tx_tr, np.zeros(tx_tr.shape[1]), 3_000_000, gamma_sgd)
print(f"{'SGD 3000000':<14} | loss {loss_sgd:.6f} | ||w - w_ls|| {np.linalg.norm(w_sgd - w_ls):.2e} | "
      f"val AUC {auc_score(y_va, tx_va @ w_sgd):.4f}")

reg logistic 1e-06     F1: 0.4275 ± 0.0031 | acc: 0.8701 ± 0.0030 | AUC: 0.8600 ± 0.0008 | threshold: 0.196 ± 0.008
reg logistic 1e-05     F1: 0.4274 ± 0.0031 | acc: 0.8701 ± 0.0030 | AUC: 0.8600 ± 0.0008 | threshold: 0.196 ± 0.008
reg logistic 1e-04     F1: 0.4271 ± 0.0037 | acc: 0.8701 ± 0.0036 | AUC: 0.8600 ± 0.0008 | threshold: 0.197 ± 0.009
reg logistic 1e-03     F1: 0.4273 ± 0.0032 | acc: 0.8711 ± 0.0044 | AUC: 0.8600 ± 0.0007 | threshold: 0.205 ± 0.010
reg logistic 1e-02     F1: 0.4227 ± 0.0019 | acc: 0.8649 ± 0.0057 | AUC: 0.8582 ± 0.0005 | threshold: 0.238 ± 0.009

least squares  | loss 0.033336 | val AUC 0.8527
GD 3000        | loss 0.033337 | ||w - w_ls|| 4.03e-02 | val AUC 0.8527
SGD 3000000    | loss 0.033407 | ||w - w_ls|| 4.89e-02 | val AUC 0.8526


### Summary of Cross-Validated Performance (Table I)

| Model | $F_1$ | AUC |
|---|---|---|
| Always positive | 0.162 | 0.500 |
| Logistic (age, sex only) | 0.294 | 0.753 |
| Least squares (GD/SGD converge to it) | 0.4140 | 0.8518 |
| Ridge, $\lambda = 0.1$ | 0.4142 | 0.8529 |
| **Logistic** | **0.4275** | **0.8600** |
| Regularized logistic, $\lambda = 10^{-4}$ | 0.4271 | 0.8600 |

Logistic regression outperformed least squares by 0.0135 F1 and 0.0082 AUC. Ridge increased least-squares AUC by 0.0011, but regularization provided no clear F1 improvement for either model family. We therefore retained unregularized logistic regression.


## 4. Class Rebalancing — Positive Weighting & Negative Undersampling

We test weighting positive samples by $\alpha \in \{2, 5, 10.3\}$ (where $\alpha = (1-p)/p \approx 10.3$ equalizes class totals) and undersampling negative samples at rate $r = 1/\alpha$.

Neither weighting positive samples by $\alpha \in \{2, 5, 10.3\}$ nor retaining negatives with probability $1/\alpha$ improved mean validation $F_1$. Weighting increased the selected thresholds from $0.196$ to $0.333$, $0.554$ and $0.734$, while AUC changed only slightly. The strongest negative undersampling reduced AUC.


In [8]:
def weighted_gram(tx, c, chunk=20_000):
    """X^T C X with C = diag(c), accumulated in chunks."""
    g = np.zeros((tx.shape[1], tx.shape[1]))
    for s in range(0, len(tx), chunk):
        block = tx[s:s + chunk]
        g += block.T @ (block * c[s:s + chunk, None])
    return g


def weighted_logistic_gd(y, tx, c, max_iters, gamma):
    """Gradient descent on the weighted mean logistic loss."""
    w = np.zeros(tx.shape[1])
    for _ in range(max_iters):
        w -= gamma * tx.T @ (c * (sigmoid(tx @ w) - y))
    return w


def make_weighted_logreg(alpha, max_iters=1000):
    """Final preprocessing + weighted logistic regression (weight alpha on positives)."""
    def train_and_predict(x_tr, y_tr, x_va):
        prep = pp.fit_preprocessing(x_tr, pp.TAU, nominal_idx)
        tx_tr, tx_va = pp.transform(x_tr, prep), pp.transform(x_va, prep)
        c = np.where(y_tr == 1, alpha, 1.0)
        c /= c.sum()
        L = np.linalg.eigvalsh(weighted_gram(tx_tr, c)).max() / 4
        return sigmoid(tx_va @ weighted_logistic_gd(y_tr, tx_tr, c, max_iters, 1 / L))
    return train_and_predict


results6 = {"logistic": results5["logistic"]}
balanced = (1 - y01.mean()) / y01.mean()
ref = results6["logistic"]
for alpha in [2, 5, balanced]:
    name = f"weighted {alpha:.1f}"
    results6[name] = cross_validate(xr_train, y01, folds, make_weighted_logreg(alpha))
    print_cv(name, *results6[name])
    d_f1, d_auc = results6[name][0] - ref[0], results6[name][2] - ref[2]
    print(f"{'':<22} vs logistic -> dF1: {d_f1.mean():+.4f} ({d_f1.mean() / corrected_se(d_f1, K):+.1f} SE) | "
          f"dAUC: {d_auc.mean():+.4f} ({d_auc.mean() / corrected_se(d_auc, K):+.1f} SE)")

weighted 2.0           F1: 0.4271 ± 0.0043 | acc: 0.8714 ± 0.0043 | AUC: 0.8602 ± 0.0009 | threshold: 0.333 ± 0.015
                       vs logistic -> dF1: -0.0004 (-0.4 SE) | dAUC: +0.0002 (+2.6 SE)
weighted 5.0           F1: 0.4274 ± 0.0032 | acc: 0.8698 ± 0.0022 | AUC: 0.8603 ± 0.0009 | threshold: 0.554 ± 0.010
                       vs logistic -> dF1: -0.0001 (-0.1 SE) | dAUC: +0.0003 (+2.2 SE)
weighted 10.3          F1: 0.4264 ± 0.0052 | acc: 0.8738 ± 0.0050 | AUC: 0.8602 ± 0.0009 | threshold: 0.734 ± 0.014
                       vs logistic -> dF1: -0.0011 (-0.7 SE) | dAUC: +0.0003 (+1.6 SE)


In [9]:
def make_undersampled_logreg(keep, max_iters=1000, seed=SEED):
    """Final preprocessing + logistic regression on all positives and a fraction `keep` of negatives."""
    def train_and_predict(x_tr, y_tr, x_va):
        prep = pp.fit_preprocessing(x_tr, pp.TAU, nominal_idx)
        rng = np.random.default_rng(seed)
        sub = (y_tr == 1) | (rng.random(len(y_tr)) < keep)
        tx_sub = pp.transform(x_tr[sub], prep)
        L = np.linalg.eigvalsh(tx_sub.T @ tx_sub / len(tx_sub)).max() / 4
        w, _ = logistic_regression(y_tr[sub], tx_sub, np.zeros(tx_sub.shape[1]), max_iters, 1 / L)
        return sigmoid(pp.transform(x_va, prep) @ w)
    return train_and_predict


for alpha in [2, 5, balanced]:
    name = f"undersampled 1/{alpha:.1f}"
    results6[name] = cross_validate(xr_train, y01, folds, make_undersampled_logreg(1 / alpha))
    print_cv(name, *results6[name])
    d_f1, d_auc = results6[name][0] - ref[0], results6[name][2] - ref[2]
    print(f"{'':<22} vs logistic -> dF1: {d_f1.mean():+.4f} ({d_f1.mean() / corrected_se(d_f1, K):+.1f} SE) | "
          f"dAUC: {d_auc.mean():+.4f} ({d_auc.mean() / corrected_se(d_auc, K):+.1f} SE)")

undersampled 1/2.0     F1: 0.4274 ± 0.0038 | acc: 0.8694 ± 0.0028 | AUC: 0.8600 ± 0.0009 | threshold: 0.329 ± 0.010
                       vs logistic -> dF1: -0.0001 (-0.1 SE) | dAUC: +0.0001 (+1.0 SE)
undersampled 1/5.0     F1: 0.4269 ± 0.0044 | acc: 0.8736 ± 0.0039 | AUC: 0.8599 ± 0.0009 | threshold: 0.569 ± 0.011
                       vs logistic -> dF1: -0.0006 (-0.6 SE) | dAUC: -0.0001 (-0.5 SE)
undersampled 1/10.3    F1: 0.4263 ± 0.0036 | acc: 0.8736 ± 0.0049 | AUC: 0.8593 ± 0.0010 | threshold: 0.738 ± 0.011
                       vs logistic -> dF1: -0.0012 (-1.0 SE) | dAUC: -0.0006 (-2.1 SE)


## 5. Bias–Variance Analysis & Non-Linear Feature Expansions

On fold 0, with preprocessing held fixed, training and validation AUC were 0.8611 and 0.8612 at full training size, suggesting a small generalization gap for this configuration.

We therefore tested three non-linear feature expansions:
- **Quintile-binning** continuous and count variables ($\Delta\text{AUC} = +0.0001$).
- **Interacting all features with age** (552 non-bias features): improves ranking ($+0.0014$ AUC, $4.9\,\widehat{\mathrm{SE}}$), but the $F_1$ gain ($+0.0010$, $0.6\,\widehat{\mathrm{SE}}$) is below our selection threshold ($0.002$ and $2\,\widehat{\mathrm{SE}}$).
- **Adding sex interactions** (828 non-bias features): provides no further F1 improvement beyond age interactions.

In [10]:
def logistic_loss(y, scores):
    """Mean logistic loss for labels in {0, 1} and linear scores."""
    return np.mean(np.log1p(np.exp(scores)) - y * scores)


# Learning curve on Fold 0 (5% to 100% of the training split)
perm = np.random.default_rng(SEED).permutation(tr)
for frac in [0.05, 0.1, 0.25, 0.5, 1.0]:
    sub = perm[:int(frac * len(tr))]
    tx_sub = pp.transform(xr_train[sub], prep0)
    L = np.linalg.eigvalsh(tx_sub.T @ tx_sub / len(tx_sub)).max() / 4
    w, _ = logistic_regression(y01[sub], tx_sub, np.zeros(tx_sub.shape[1]), 1000, 1 / L)
    s_sub, s_va = tx_sub @ w, tx_va @ w
    print(f"{frac:>4.0%} ({len(sub):>6} rows) | loss train {logistic_loss(y01[sub], s_sub):.4f} "
          f"val {logistic_loss(y01[va], s_va):.4f} | AUC train {auc_score(y01[sub], s_sub):.4f} "
          f"val {auc_score(y01[va], s_va):.4f}")

  5% ( 13125 rows) | loss train 0.2082 val 0.2302 | AUC train 0.8682 val 0.8476
 10% ( 26250 rows) | loss train 0.2150 val 0.2257 | AUC train 0.8629 val 0.8541
 25% ( 65627 rows) | loss train 0.2186 val 0.2229 | AUC train 0.8625 val 0.8585
 50% (131254 rows) | loss train 0.2202 val 0.2223 | AUC train 0.8618 val 0.8597
100% (262508 rows) | loss train 0.2211 val 0.2215 | AUC train 0.8611 val 0.8612


In [11]:
def interaction_features(tx, cols):
    """Products of the features in positions `cols` with all features (bias excluded)."""
    x = tx[:, 1:]
    return np.hstack([x * x[:, [c]] for c in cols])


def make_interaction_logreg(partners, max_iters=1000):
    """Final preprocessing + standardized interaction terms with `partners`."""
    def train_and_predict(x_tr, y_tr, x_va):
        prep = pp.fit_preprocessing(x_tr, pp.TAU, nominal_idx)
        names = pp.feature_names(prep, kept_names)
        cols = [int(np.where(names == p)[0][0]) for p in partners]
        tx_tr, tx_va = pp.transform(x_tr, prep), pp.transform(x_va, prep)
        z_tr, z_va = interaction_features(tx_tr, cols), interaction_features(tx_va, cols)
        mean, std = z_tr.mean(axis=0), z_tr.std(axis=0)
        keep = std > 0
        tx_tr = np.hstack([tx_tr, (z_tr[:, keep] - mean[keep]) / std[keep]])
        tx_va = np.hstack([tx_va, (z_va[:, keep] - mean[keep]) / std[keep]])
        L = np.linalg.eigvalsh(tx_tr.T @ tx_tr / len(tx_tr)).max() / 4
        w, _ = logistic_regression(y_tr, tx_tr, np.zeros(tx_tr.shape[1]), max_iters, 1 / L)
        return sigmoid(tx_va @ w)
    return train_and_predict


BINNED = pp.WINSOR + ["PHYSHLTH", "MENTHLTH", "POORHLTH", "NUMADULT",
                      "HHADULT", "AVEDRNK2", "DRNK3GE5", "MAXDRNKS", "JOINPAIN"]
binned_idx = [int(np.where(kept_names == n)[0][0]) for n in BINNED]


def bin_dummies(x, cols, edges):
    """One dummy per quantile bin of each column (lowest bin as reference)."""
    out = []
    for c, e in zip(cols, edges):
        v = x[:, c]
        d = (np.digitize(v, e)[:, None] == np.arange(1, len(e) + 1)).astype(float)
        d[np.isnan(v)] = 0
        out.append(d)
    return np.hstack(out)


def make_binned_logreg(n_bins=5, max_iters=1000):
    """Final preprocessing + quintile-bin dummies for continuous and count columns."""
    def train_and_predict(x_tr, y_tr, x_va):
        prep = pp.fit_preprocessing(x_tr, pp.TAU, nominal_idx)
        cols = [c for c in binned_idx if c in prep["num_cols"]]
        edges = [np.unique(np.nanquantile(x_tr[:, c], np.linspace(0, 1, n_bins + 1)[1:-1])) for c in cols]
        z_tr, z_va = bin_dummies(x_tr, cols, edges), bin_dummies(x_va, cols, edges)
        mean, std = z_tr.mean(axis=0), z_tr.std(axis=0)
        keep = std > 0
        tx_tr = np.hstack([pp.transform(x_tr, prep), (z_tr[:, keep] - mean[keep]) / std[keep]])
        tx_va = np.hstack([pp.transform(x_va, prep), (z_va[:, keep] - mean[keep]) / std[keep]])
        L = np.linalg.eigvalsh(tx_tr.T @ tx_tr / len(tx_tr)).max() / 4
        w, _ = logistic_regression(y_tr, tx_tr, np.zeros(tx_tr.shape[1]), max_iters, 1 / L)
        return sigmoid(tx_va @ w)
    return train_and_predict


results7 = {"logistic": results5["logistic"]}
for name, fn in [
    ("age interactions", make_interaction_logreg(["_AGE80"])),
    ("age + sex interactions", make_interaction_logreg(["_AGE80", "SEX"])),
    ("binned", make_binned_logreg()),
]:
    results7[name] = cross_validate(xr_train, y01, folds, fn)
    print_cv(name, *results7[name])
    d_f1, d_auc = results7[name][0] - ref[0], results7[name][2] - ref[2]
    print(f"{'':<22} vs logistic -> dF1: {d_f1.mean():+.4f} ({d_f1.mean() / corrected_se(d_f1, K):+.1f} SE) | "
          f"dAUC: {d_auc.mean():+.4f} ({d_auc.mean() / corrected_se(d_auc, K):+.1f} SE)")

age interactions       F1: 0.4285 ± 0.0014 | acc: 0.8736 ± 0.0074 | AUC: 0.8614 ± 0.0008 | threshold: 0.207 ± 0.016
                       vs logistic -> dF1: +0.0010 (+0.6 SE) | dAUC: +0.0014 (+4.9 SE)
age + sex interactions F1: 0.4284 ± 0.0017 | acc: 0.8699 ± 0.0043 | AUC: 0.8614 ± 0.0006 | threshold: 0.201 ± 0.008
                       vs logistic -> dF1: +0.0009 (+0.6 SE) | dAUC: +0.0015 (+4.9 SE)
binned                 F1: 0.4271 ± 0.0047 | acc: 0.8711 ± 0.0034 | AUC: 0.8600 ± 0.0009 | threshold: 0.199 ± 0.009
                       vs logistic -> dF1: -0.0004 (-0.3 SE) | dAUC: +0.0001 (+0.5 SE)


## 6. Ablation Study (Table II) & Conclusion

We remove one component at a time from the final model (unregularized logistic regression, $\tau = 0.8$, one-hot encoding, recoded survey values, and tuned threshold) and evaluate the paired change in $F_1$ and AUC.

In [12]:
def make_final_logreg(tau=pp.TAU, indicators=True, max_iters=1000):
    """Final logistic regression pipeline with configurable tau and missingness indicators."""
    def train_and_predict(x_tr, y_tr, x_va):
        prep = pp.fit_preprocessing(x_tr, tau, nominal_idx, indicators=indicators)
        tx_tr, tx_va = pp.transform(x_tr, prep), pp.transform(x_va, prep)
        L = np.linalg.eigvalsh(tx_tr.T @ tx_tr / len(tx_tr)).max() / 4
        w, _ = logistic_regression(y_tr, tx_tr, np.zeros(tx_tr.shape[1]), max_iters, 1 / L)
        return sigmoid(tx_va @ w)
    return train_and_predict


x_raw = pp.apply_recoding(x_train, col_names, kept_names, {n: ([], []) for n in kept_names})

results8 = {"final": results5["logistic"]}
ablations = {
    "threshold 0.5": (xr_train, make_final_logreg(), 0.5),
    "no recoding": (x_raw, make_final_logreg(), None),
    "tau 0.3": (xr_train, make_final_logreg(tau=0.3), None),
    "no missingness indicators": (xr_train, make_final_logreg(indicators=False), None),
}
ref = results8["final"]
for name, (x, model, threshold) in ablations.items():
    results8[name] = cross_validate(x, y01, folds, model, threshold=threshold)
    print_cv(name, *results8[name])
    d_f1, d_auc = results8[name][0] - ref[0], results8[name][2] - ref[2]
    se_auc = f"{d_auc.mean() / corrected_se(d_auc, K):+.1f} SE" if np.any(d_auc != 0) else "0 SE"
    print(f"{'':<22} vs final -> dF1: {d_f1.mean():+.4f} ({d_f1.mean() / corrected_se(d_f1, K):+.1f} SE) | "
          f"dAUC: {d_auc.mean():+.4f} ({se_auc})")

threshold 0.5          F1: 0.2336 ± 0.0048 | acc: 0.9148 ± 0.0003 | AUC: 0.8600 ± 0.0008 | threshold: 0.500 ± 0.000
                       vs final -> dF1: -0.1939 (-42.8 SE) | dAUC: +0.0000 (0 SE)
no recoding            F1: 0.4102 ± 0.0035 | acc: 0.8668 ± 0.0053 | AUC: 0.8489 ± 0.0008 | threshold: 0.203 ± 0.009
                       vs final -> dF1: -0.0173 (-10.2 SE) | dAUC: -0.0111 (-158.2 SE)
tau 0.3                F1: 0.4255 ± 0.0031 | acc: 0.8650 ± 0.0061 | AUC: 0.8592 ± 0.0009 | threshold: 0.187 ± 0.012
                       vs final -> dF1: -0.0020 (-1.8 SE) | dAUC: -0.0007 (-5.3 SE)
no missingness indicators F1: 0.4275 ± 0.0040 | acc: 0.8694 ± 0.0030 | AUC: 0.8595 ± 0.0010 | threshold: 0.194 ± 0.005
                       vs final -> dF1: -0.0000 (-0.0 SE) | dAUC: -0.0005 (-2.5 SE)


### Ablation Summary (Table II)

| Component removed | $F_1$ | $\Delta F_1$ | $\Delta\text{AUC}$ |
|---|---|---|---|
| — (Final logistic model) | 0.4275 ± 0.0031 | — | — |
| Tuned threshold ($0.5$ instead) | 0.2336 ± 0.0048 | $-0.1939$ ($-42.8\,\widehat{\mathrm{SE}}$) | $0$ |
| Recoding of special codes | 0.4102 ± 0.0035 | $-0.0173$ ($-10.2\,\widehat{\mathrm{SE}}$) | $-0.0111$ ($-158\,\widehat{\mathrm{SE}}$) |
| Logistic loss (least squares) | 0.4140 ± 0.0020 | $-0.0135$ ($-6.8\,\widehat{\mathrm{SE}}$) | $-0.0082$ ($-41\,\widehat{\mathrm{SE}}$) |
| One-hot encoding | 0.4250 ± 0.0038 | $-0.0025$ ($-3.6\,\widehat{\mathrm{SE}}$) | $-0.0013$ ($-6.5\,\widehat{\mathrm{SE}}$) |
| $\tau = 0.8$ ($\tau = 0.3$ instead) | 0.4255 ± 0.0031 | $-0.0020$ ($-1.8\,\widehat{\mathrm{SE}}$) | $-0.0007$ ($-5.3\,\widehat{\mathrm{SE}}$) |
| Missingness indicators | 0.4275 ± 0.0040 | $-0.0000$ ($-0.0\,\widehat{\mathrm{SE}}$) | $-0.0005$ ($-2.5\,\widehat{\mathrm{SE}}$) |

### Conclusion
- **Threshold tuning** increased mean validation $F_1$ from $0.2336$ at $t = 0.5$ to $0.4275$, while accuracy decreased from $0.9148$ to $0.8701$. The final submission uses $t = 0.20$ and achieved AIcrowd $F_1 = 0.439$.
- **Recoding survey codes** ($-0.0173\,F_1$ when omitted) matters more than the choice of ML algorithm itself.
- **Regularization, class weighting, and undersampling** provide no $F_1$ gain, and **age interactions** improve ranking ($+0.0014$ AUC) without meeting our $F_1$ selection criterion, so we retain the simpler unregularized logistic regression model.

### References
1. Centers for Disease Control and Prevention, *BRFSS 2015 Codebook Report*, 2016.
2. C. Nadeau and Y. Bengio, "Inference for the Generalization Error," *Machine Learning*, vol. 52, no. 3, 2003.